# RAG Minecraft

## Configuration Gemini API

In [7]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers import MultiVectorRetriever
import pickle
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time

/var/folders/n6/53m8gkg13qvb_3qz8npdf9lm0000gn/T/ipykernel_23610/3770355338.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

### Config sources

In [59]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [9]:
scraper = cloudscraper.create_scraper()

In [10]:
def write_csv(file_name, paragraphs):
    # Create parent folder automatically if it does not exist
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

### Scraping Utils

#### Wikipedia

In [125]:
from langchain_core.documents import Document

def parse_wikipedia_sections(soup, source):

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_text = []

    def save_section():
        nonlocal current_text

        text = "\n".join(current_text).strip()

        # ignorer sections vides
        if not text:
            current_text = []
            return

        title_parts = [x for x in [h2, h3, h4] if x]
        section_title = " > ".join(title_parts)

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": source,
                    "section": section_title
                }
            )
        )

        current_text = []

    for tag in soup.find_all(["h2", "h3", "h4", "p", "li"]):

        txt = tag.get_text(" ", strip=True)

        if not txt:
            continue

        # arrêt avant les sections inutiles
        if tag.name == "h2" and any(
            x in txt
            for x in [
                "Références",
                "Notes et références",
                "Voir aussi",
                "Liens externes",
                "Accueil",
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h3" and any(
            x in txt
            for x in [
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h2":

            save_section()

            h2 = txt
            h3 = None
            h4 = None

        elif tag.name == "h3":

            save_section()

            h3 = txt
            h4 = None

        elif tag.name == "h4":

            save_section()

            h4 = txt

        else:
            current_text.append(txt)

    save_section()

    return docs

#### Fandom

In [65]:
from bs4 import BeautifulSoup

def get_section_links(soup, section_name):

    section = soup.find(
        lambda tag:
        tag.name in ["h2", "h3"]
        and section_name in tag.get_text()
    )

    if not section:
        return []

    links = []

    node = section.find_next_sibling()

    while node:

        if node.name in ["h2", "h3"]:
            break

        for a in node.find_all("a", title=True):

            title = a["title"].strip()

            if ":" in title:
                continue

            links.append(title)

        node = node.find_next_sibling()

    return links

In [70]:
import re

def clean_wiki_text(text: str) -> str:

    # espaces multiples
    text = re.sub(r"[ \t]+", " ", text)

    # lignes vides multiples
    text = re.sub(r"\n{3,}", "\n\n", text)

    # supprimer certains marqueurs
    text = re.sub(r"\[Version .*?\]", "", text)

    # supprimer espaces en début/fin
    text = text.strip()

    return text

In [69]:
def normalize_headings(text):
    text = re.sub(
        r"\n==\s*(.*?)\s*==\n",
        r"\nSECTION: \1\n",
        text
    )

    return text

In [71]:
STOP_SECTIONS = [
    "== Références ==",
    "== Notes diverses ==",
    "== Voir aussi ==",
]

def truncate_useless_sections(text):
    for section in STOP_SECTIONS:
        pos = text.find(section)

        if pos != -1:
            text = text[:pos]

    return text

### Wikipedia

In [123]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(
        url,
        params=params,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")
    docs = parse_wikipedia_sections(soup, f"wikipedia:{title}" )
    
    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        } 
        for doc in docs
    ]

    write_csv(
        f"files/wikipedia.csv",
        docs
    )
    return docs


### Fandom

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote
from langchain_core.documents import Document


def scrape_fandom(url: str):

    parsed = urlparse(url)

    if "/wiki/" not in parsed.path:
        return None

    page_name = unquote(
        parsed.path.split("/wiki/")[-1]
    )


    parts = parsed.path.split("/")
    lang = parts[1] if len(parts) > 2 else ""

    api_url = (
        f"{parsed.scheme}://{parsed.netloc}/{lang}/api.php"
    )

    params = {
        "action": "query",
        "prop": "extracts",
        "titles": page_name,
        "format": "json",
        "explaintext": True,
        "redirects": 1,
    }

    response = requests.get(
        api_url,
        params=params,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30,
    )

    data = response.json()

    pages = data.get("query", {}).get("pages", {})

    if not pages:
        return None

    page = next(iter(pages.values()))

    text = page.get("extract", "")

    if not text:
        return None

    paragraphs = [
        p.strip()
        for p in text.split("\n")
        if p.strip()
    ]

    text = page.get("extract", "")

    text = truncate_useless_sections(text)

    text = clean_wiki_text(text)

    text = normalize_headings(text)

    print(f"API OK: {page_name}")
    write_csv(f"files/{page_name}.csv", [text])

    return Document(
        page_content=text,
        metadata={
            "title": page.get("title", page_name),
            "source": url,
        }
    )

### Build dataset

In [130]:
docs = []

# Wikipedia
for page in WIKI_PAGES:
    try:
        docs.extend(scrape_wikipedia(page))
        print("WIKI OK:", page)
    except Exception as e:
        print("WIKI ERROR:", page, e)

# Fandom
# for url in FANDOM_URLS:
#     try:
#         doc = scrape_fandom(url)
#         if doc:
#             docs.append(doc)
#             print("FANDOM OK:", url)
#     except Exception as e:
#         print("FANDOM ERROR:", url, e)

print("\nTOTAL DOCS:", len(docs))
for i, x in enumerate(docs):
    print(x["section"], ":", len(x["page_content"]))

WIKI OK: Minecraft

TOTAL DOCS: 17
 : 1662
Trame : 482
Système de jeu : 2550
Système de jeu > Monde : 2789
Système de jeu > Créatures : 3346
Système de jeu > Outils, armes et armures : 2235
Système de jeu > Redstone : 402
Système de jeu > Modes de jeu > Mode survie : 1207
Système de jeu > Modes de jeu > Mode créatif : 1205
Système de jeu > Modes de jeu > Mode aventure : 1116
Système de jeu > Modes de jeu > Mode Hardcore : 803
Système de jeu > Modes de jeu > Mode spectateur : 955
Système de jeu > Multijoueur > Principe : 839
Système de jeu > Multijoueur > Serveurs : 969
Système de jeu > Multijoueur > Realms et Realms Plus : 335
Système de jeu > Multijoueur > Changement de pseudonyme : 470
Développement > Historique : 8291


Chunking

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

# i = 0
# while i < 3:
#     print("Chunk", i)
#     print(split_docs[i])
#     i +=1

print("CHUNKS:", len(split_docs))

Chunk 0
page_content='Minecraft plonge le joueur dans un monde créé de manière procédurale, composé de voxels représentant différents matériaux ( terre , pierre , eau , fer , charbon , etc. ). Le monde est formé de diverses structures ( arbres , cavernes , montagnes , villages , etc. ) et est peuplé par des animaux ( vaches , moutons , etc. ) ainsi que des monstres ( zombies , araignées , squelettes , etc. ). Le joueur peut modifier son monde à volonté, soit dans le but de survivre, soit pour créer.

Le joueur est représenté sous une forme humanoïde de deux blocs de haut et porte un pseudonyme demandé lors de l'achat du jeu. Son apparence (« skin » en anglais, soit « peau »), peut être personnalisée sur le site officiel ou dans le jeu, mais il existe des apparences par défaut. Jusqu'en 2015 [ 1 ] , seule l'apparence nommée Steve, un personnage masculin habillé d'un jean bleu et d'une veste turquoise, est disponible [ 2 ] . En 2015 est introduite Alex, portant un jean marron, un chandai

Enrchich

In [75]:
def enrich_with_source(docs):
    enriched = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        # Extraire le nom de la page depuis l'URL comme "pseudo-titre"
        if "wiki/" in src:
            page_title = src.split("wiki/")[-1].replace("_", " ")
        elif "wikipedia:" in src:
            page_title = src.split("wikipedia:")[-1].replace("_", " ")
        else:
            page_title = src

        # Enrichissement du chunk : on ajoute le contexte AVANT le texte
        # → la similarité vectorielle bénéficie du titre + type de source
        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n"
            f"TOPIC: {page_title}\n\n"   # ← nouveauté
            f"{d.page_content}"
        )
        enriched.append(d)
    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [25]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

#### Store data using chroma

In [27]:
import shutil
import os
path = "./chroma_minecraft_db"

if os.path.exists(path):
    try:
        shutil.rmtree(path)
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")

In [ ]:
# --- ÉTAPE 1 : Générer un résumé par chunk ---

summarize_chain = (
    PromptTemplate.from_template(
        """
Résume cette page Minecraft pour un moteur de recherche sémantique.

Mentionne explicitement :
- les principaux concepts ;
- les créatures ;
- les objets ;
- les structures ;
- les biomes ;
- les mécaniques de jeu.

Le résumé doit faire entre 100 et 200 mots.

{doc}
"""
    )
    | llm
    | StrOutputParser()
)


summaries = []
for doc in docs:
    doc.
    # summaries.append(summary)
    write_csv("summaries/")

In [ ]:


# --- ÉTAPE 2 : Associer chaque résumé à son chunk original ---
# On crée un ID unique par chunk pour faire le lien
# résumé (dans Chroma) <-> chunk original (dans le docstore)
import uuid
doc_ids = [str(uuid.uuid4()) for _ in split_docs]

# Les résumés sont les documents indexés dans Chroma
# On colle l'ID dans les metadata pour retrouver l'original
summary_docs = [
    Document(
        page_content=summaries[i],
        metadata={"doc_id": doc_ids[i]}  # clé de liaison
    )
    for i in range(len(split_docs))
]

# --- ÉTAPE 3 : Construire le retriever ---
# InMemoryByteStore = le "tiroir" qui stocke les chunks originaux
# Le MultiVectorRetriever fait le pont :
#   1. cherche dans Chroma (résumés)
#   2. récupère l'ID du résumé trouvé
#   3. va chercher le chunk original dans le docstore
#   4. retourne le chunk original au LLM


vectorstore = Chroma(
    persist_directory="./chroma_minecraft_multivec",
    embedding_function=gemini_embeddings
)
vectorstore.add_documents(summary_docs)

store = InMemoryByteStore()  # stockage des chunks originaux en mémoire

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,   # Chroma contient les résumés
    byte_store=store,          # le store contient les chunks bruts
    id_key="doc_id",           # la clé qui fait le lien
)

# On peuple le docstore avec les chunks originaux
retriever.docstore.mset(list(zip(doc_ids, split_docs)))

In [ ]:
# Check the number of chunks generated after text splitting.
print("split_docs:", len(split_docs))

# Check the number of documents actually stored in the Chroma vector database.
# If this number is equal to len(split_docs), the database was built without missing or duplicated chunks.
print("chroma:", vectorstore._collection.count())

#### Create a retriever using Chroma

In [ ]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

## Generator

#### Create prompt templates

In [ ]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

#### Create a stuff documents chain

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [ ]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        nouvelle_requete = str(llm_rewrite.invoke(rewrite_prompt).content)
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [ ]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

In [ ]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft? Décris les."))

In [ ]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft, et décris les tous."))

In [ ]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)